# Pipeline ML — Classification par Features Extraites (HOG + LBP)

Classification de radiographies pulmonaires masquées (COVID-19) à l'aide de descripteurs visuels **HOG** (Histogram of Oriented Gradients) et **LBP** (Local Binary Patterns), combinés à des classifieurs de Machine Learning classiques.

**Dataset** : 21 165 images 256×256 en niveaux de gris, 4 classes (COVID, Lung_Opacity, Normal, Viral Pneumonia)

In [1]:
import os
import time
import warnings

import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm

from skimage.feature import (
    hog, local_binary_pattern,
    graycomatrix, graycoprops,  # GLCM (Haralick-like)
)
from skimage.filters import gabor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Imports OK')

Imports OK


In [2]:
# ── Configuration ──
DATASET_PATH = '../data/processed/masked_full_dataset_256_256_L'
IMG_SIZE = 256
RANDOM_SEED = 42
CLASS_NAMES = sorted(os.listdir(DATASET_PATH))

# ── Sélection des techniques de Feature Extraction ──
# Passer à True/False pour activer/désactiver chaque technique
FEATURES_CONFIG = {
    'HOG':            True,   # Histogram of Oriented Gradients — contours & structures (~34 596 features)
    'LBP':            True,   # Local Binary Patterns — micro-textures (~26 features)
    'GLCM':           True,  # Gray-Level Co-occurrence Matrix — textures statistiques (~20 features)
    'Gabor':          True,  # Filtres de Gabor — textures multi-échelles & orientations (~40 features)
    'Histogram':      True,  # Histogramme de luminosité des pixels (~64 features)
    'Hu Moments':     True,  # Moments de Hu — forme globale invariante (~7 features)
    'Edge Histogram': True,  # Histogramme des orientations de contours Sobel (~36 features)
}

active = [k for k, v in FEATURES_CONFIG.items() if v]
print(f'Dataset : {DATASET_PATH}')
print(f'Classes : {CLASS_NAMES}')
print(f'Features actives : {active}')

Dataset : ../data/processed/masked_full_dataset_256_256_L
Classes : ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Features actives : ['HOG', 'LBP', 'GLCM', 'Gabor', 'Histogram', 'Hu Moments', 'Edge Histogram']


In [3]:
# ── Chargement du dataset ──
images = []
labels = []

for class_name in CLASS_NAMES:
    class_dir = os.path.join(DATASET_PATH, class_name)
    filenames = os.listdir(class_dir)
    print(f'{class_name}: {len(filenames)} images')
    for fname in filenames:
        img_path = os.path.join(class_dir, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(class_name)

images = np.array(images, dtype=np.uint8)
labels = np.array(labels)

print(f'\nTotal : {len(images)} images, shape = {images.shape}')
print(f'Labels : {np.unique(labels, return_counts=True)}')

COVID: 3616 images


Lung_Opacity: 6012 images
Normal: 10192 images
Viral Pneumonia: 1345 images

Total : 21165 images, shape = (21165, 256, 256)
Labels : (array(['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'], dtype='<U15'), array([ 3616,  6012, 10192,  1345]))


In [4]:
# ── Sous-échantillonnage pour dataset équilibré ──
# On prend N images par classe (= taille de la plus petite classe)
N_PER_CLASS = 10 #1345  # Viral Pneumonia

rng = np.random.RandomState(RANDOM_SEED)
balanced_indices = []

for class_name in CLASS_NAMES:
    class_indices = np.where(labels == class_name)[0]
    selected = rng.choice(class_indices, size=N_PER_CLASS, replace=False)
    balanced_indices.append(selected)

balanced_indices = np.concatenate(balanced_indices)
rng.shuffle(balanced_indices)

images = images[balanced_indices]
labels = labels[balanced_indices]

print(f'Dataset équilibré : {len(images)} images ({N_PER_CLASS} par classe)')
print(f'Distribution : {np.unique(labels, return_counts=True)}')

Dataset équilibré : 40 images (10 par classe)
Distribution : (array(['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'], dtype='<U15'), array([10, 10, 10, 10]))


## Extraction de Features

Techniques disponibles (activables dans `FEATURES_CONFIG`) :

| Technique | Ce qu'elle capture | Nb features |
|-----------|-------------------|-------------|
| **HOG** | Gradients / contours / structure anatomique | ~34 596 |
| **LBP** | Micro-textures locales | ~26 |
| **GLCM** | Textures statistiques (contraste, homogénéité, corrélation, énergie) | ~20 |
| **Gabor** | Textures à différentes fréquences et orientations | ~40 |
| **Histogram** | Distribution globale d'intensité des pixels | ~64 |
| **Hu Moments** | Forme globale (invariant rotation/échelle) | 7 |
| **Edge Histogram** | Distribution des orientations de contours (Sobel) | ~36 |

In [5]:
# ── Fonctions d'extraction de features ──

def extract_hog_features(image):
    """Contours & structures via Histogram of Oriented Gradients."""
    return hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True,
    )


def extract_lbp_features(image, P=24, R=3):
    """Micro-textures via Local Binary Patterns (histogramme)."""
    lbp = local_binary_pattern(image, P=P, R=R, method='uniform')
    n_bins = P + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist


def extract_glcm_features(image):
    """Textures statistiques via Gray-Level Co-occurrence Matrix."""
    # Quantifier en 64 niveaux pour accélérer le calcul
    img_q = (image // 4).astype(np.uint8)
    distances = [1, 3]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(img_q, distances=distances, angles=angles, levels=64, symmetric=True, normed=True)
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
    feats = []
    for prop in props:
        feats.append(graycoprops(glcm, prop).ravel())
    return np.concatenate(feats)


def extract_gabor_features(image):
    """Textures multi-échelles via filtres de Gabor."""
    feats = []
    frequencies = [0.05, 0.1, 0.2, 0.3, 0.4]
    thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    for freq in frequencies:
        for theta in thetas:
            filt_real, filt_imag = gabor(image, frequency=freq, theta=theta)
            feats.extend([filt_real.mean(), filt_real.std()])
    return np.array(feats)


def extract_histogram_features(image, n_bins=64):
    """Distribution globale d'intensité des pixels."""
    hist, _ = np.histogram(image.ravel(), bins=n_bins, range=(0, 256), density=True)
    return hist


def extract_hu_moments(image):
    """Forme globale via les 7 moments de Hu (log-transformés)."""
    moments = cv2.moments(image)
    hu = cv2.HuMoments(moments).flatten()
    # Log-transform pour stabiliser les valeurs (éviter les très grands écarts)
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
    return hu_log


def extract_edge_histogram(image, n_bins=36):
    """Distribution des orientations de contours (Sobel)."""
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    angle = np.arctan2(sobel_y, sobel_x) * 180 / np.pi  # -180 à 180
    # Histogramme des angles pondéré par la magnitude
    hist, _ = np.histogram(angle.ravel(), bins=n_bins, range=(-180, 180), weights=magnitude.ravel(), density=True)
    return hist


# ── Registre des extracteurs ──
EXTRACTORS = {
    'HOG':            extract_hog_features,
    'LBP':            extract_lbp_features,
    'GLCM':           extract_glcm_features,
    'Gabor':          extract_gabor_features,
    'Histogram':      extract_histogram_features,
    'Hu Moments':     extract_hu_moments,
    'Edge Histogram': extract_edge_histogram,
}


def extract_selected_features(image):
    """Concaténer les features des techniques activées dans FEATURES_CONFIG."""
    parts = []
    for name, enabled in FEATURES_CONFIG.items():
        if enabled:
            parts.append(EXTRACTORS[name](image))
    return np.concatenate(parts)


# ── Vérification sur une image ──
sample_feat = extract_selected_features(images[0])
print(f'Features sélectionnées : {[k for k, v in FEATURES_CONFIG.items() if v]}')
print(f'Vecteur total : {sample_feat.shape[0]} features\n')

# Détail par technique
for name, enabled in FEATURES_CONFIG.items():
    if enabled:
        f = EXTRACTORS[name](images[0])
        print(f'  {name:18s} : {f.shape[0]:>6} features')

Features sélectionnées : ['HOG', 'LBP', 'GLCM', 'Gabor', 'Histogram', 'Hu Moments', 'Edge Histogram']
Vecteur total : 34809 features

  HOG                :  34596 features
  LBP                :     26 features
  GLCM               :     40 features
  Gabor              :     40 features
  Histogram          :     64 features
  Hu Moments         :      7 features
  Edge Histogram     :     36 features


In [6]:
# ── Visualisation des features sur un exemple ──
from matplotlib.colors import LogNorm

sample_img = images[80]

n_active = sum(FEATURES_CONFIG.values())
fig, axes = plt.subplots(1, 1 + n_active, figsize=(5 * (1 + n_active), 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

axes[0].imshow(sample_img, cmap='gray')
axes[0].set_title('Image originale')
axes[0].axis('off')

col = 1
if FEATURES_CONFIG.get('HOG'):
    _, hog_image = hog(
        sample_img, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), visualize=True, feature_vector=True,
    )
    axes[col].imshow(hog_image, cmap='gray')
    axes[col].set_title('HOG')
    axes[col].axis('off')
    col += 1

if FEATURES_CONFIG.get('LBP'):
    lbp_image = local_binary_pattern(sample_img, P=24, R=3, method='uniform')
    axes[col].imshow(lbp_image, cmap='gray')
    axes[col].set_title('LBP')
    axes[col].axis('off')
    col += 1

if FEATURES_CONFIG.get('GLCM'):
    img_q = (sample_img // 4).astype(np.uint8)
    glcm = graycomatrix(img_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)
    glcm_2d = glcm[:, :, 0, 0].astype(float)
    glcm_2d[glcm_2d == 0] = np.nan  # masquer les zéros pour le log
    axes[col].imshow(glcm_2d, cmap='hot', interpolation='nearest',
                     norm=LogNorm(vmin=np.nanmin(glcm_2d[glcm_2d > 0]), vmax=np.nanmax(glcm_2d)))
    axes[col].set_title('GLCM (d=1, θ=0) [log]')
    axes[col].axis('off')
    col += 1

if FEATURES_CONFIG.get('Gabor'):
    filt_real, _ = gabor(sample_img, frequency=0.1, theta=0)
    axes[col].imshow(filt_real, cmap='gray')
    axes[col].set_title('Gabor (f=0.1, θ=0)')
    axes[col].axis('off')
    col += 1

if FEATURES_CONFIG.get('Edge Histogram'):
    sobel_x = cv2.Sobel(sample_img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(sample_img, cv2.CV_64F, 0, 1, ksize=3)
    edges = np.sqrt(sobel_x**2 + sobel_y**2)
    axes[col].imshow(edges, cmap='gray')
    axes[col].set_title('Contours (Sobel)')
    axes[col].axis('off')
    col += 1

# Histogram et Hu Moments n'ont pas de visualisation image pertinente
for i in range(col, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

IndexError: index 80 is out of bounds for axis 0 with size 40

In [7]:
# ── Extraction sur tout le dataset ──
active = [k for k, v in FEATURES_CONFIG.items() if v]
print(f'Extraction de features ({", ".join(active)}) sur {len(images)} images...')
start = time.time()

X = []
for i, img in enumerate(tqdm(images, desc='Extraction')):
    X.append(extract_selected_features(img))

X = np.array(X)
print(f'\nFeatures extraites : X.shape = {X.shape}')
print(f'Temps total : {time.time() - start:.1f}s')

Extraction de features (HOG, LBP, GLCM, Gabor, Histogram, Hu Moments, Edge Histogram) sur 40 images...


Extraction:  42%|████▎     | 17/40 [02:22<03:12,  8.38s/it]


KeyboardInterrupt: 

## Préparation des données

In [ ]:
# ── Encodage des labels ──
le = LabelEncoder()
y = le.fit_transform(labels)

# ── Split train/test stratifié ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# ── Standardisation ──
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train : {X_train_s.shape}  |  Test : {X_test_s.shape}')
print(f'Distribution train : {np.bincount(y_train)}')
print(f'Distribution test  : {np.bincount(y_test)}')

## Visualisation des Features (PCA)

In [ ]:
# ── PCA 2D ──
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_train_s)

plt.figure(figsize=(10, 8))
for i, class_name in enumerate(le.classes_):
    mask = y_train == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=class_name, alpha=0.4, s=10)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Projection PCA 2D des features HOG+LBP')
plt.legend()
plt.tight_layout()
plt.show()

## Réduction de Dimensionnalité par PCA

On teste l'impact de la PCA sur les performances en réduisant le vecteur de ~34k features à un nombre de composantes capturant **95%** et **99%** de la variance expliquée. Cela permet de :
- **Accélérer** l'entraînement (surtout SVM, KNN)
- **Réduire le bruit** et potentiellement améliorer la généralisation
- **Comparer** les performances full features vs PCA

In [ ]:
# ── PCA : variance expliquée cumulée ──
pca_full = PCA(random_state=RANDOM_SEED)
pca_full.fit(X_train_s)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.searchsorted(cumvar, 0.95) + 1
n_99 = np.searchsorted(cumvar, 0.99) + 1

plt.figure(figsize=(10, 5))
plt.plot(cumvar, linewidth=2)
plt.axhline(0.95, color='red', linestyle='--', label=f'95% → {n_95} composantes')
plt.axhline(0.99, color='orange', linestyle='--', label=f'99% → {n_99} composantes')
plt.axvline(n_95, color='red', linestyle=':', alpha=0.5)
plt.axvline(n_99, color='orange', linestyle=':', alpha=0.5)
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance expliquée cumulée')
plt.title('Courbe de variance expliquée — PCA')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Features originales : {X_train_s.shape[1]}')
print(f'95% variance → {n_95} composantes (réduction {1 - n_95/X_train_s.shape[1]:.1%})')
print(f'99% variance → {n_99} composantes (réduction {1 - n_99/X_train_s.shape[1]:.1%})')

In [ ]:
# ── Préparation des jeux PCA 95% et 99% ──
PCA_CONFIGS = {
    f'PCA 95% ({n_95} comp.)': n_95,
    f'PCA 99% ({n_99} comp.)': n_99,
}

pca_datasets = {}
for label, n_comp in PCA_CONFIGS.items():
    pca_reducer = PCA(n_components=n_comp, random_state=RANDOM_SEED)
    X_train_pca = pca_reducer.fit_transform(X_train_s)
    X_test_pca = pca_reducer.transform(X_test_s)
    pca_datasets[label] = (X_train_pca, X_test_pca)
    print(f'{label} : train {X_train_pca.shape}, test {X_test_pca.shape}')

# Ajouter le jeu complet pour la boucle de comparaison
pca_datasets['Full features'] = (X_train_s, X_test_s)
print(f'Full features : train {X_train_s.shape}, test {X_test_s.shape}')

In [ ]:
# ── Comparaison Full vs PCA sur un sous-ensemble de modèles ──
comparison_models = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1),
    'SVM (RBF)': lambda: SVC(
        kernel='rbf', class_weight='balanced', random_state=RANDOM_SEED),
    'KNN (k=5)': lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Gradient Boosting': lambda: GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_SEED),
}

pca_results = []

for dataset_label, (X_tr, X_te) in pca_datasets.items():
    for model_name, model_fn in comparison_models.items():
        model = model_fn()
        t0 = time.time()
        model.fit(X_tr, y_train)
        train_time = time.time() - t0

        y_pred = model.predict(X_te)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        pca_results.append({
            'Features': dataset_label,
            'Modèle': model_name,
            'Accuracy': acc,
            'F1 Macro': f1,
            'Temps (s)': round(train_time, 1),
        })

    print(f'✓ {dataset_label} terminé')

df_pca = pd.DataFrame(pca_results)
print(f'\n{len(pca_results)} combinaisons évaluées')

In [ ]:
# ── Visualisation : impact de la PCA ──
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Pivot pour le grouped bar chart
pivot_acc = df_pca.pivot(index='Modèle', columns='Features', values='Accuracy')
pivot_f1 = df_pca.pivot(index='Modèle', columns='Features', values='F1 Macro')

# Ordonner les colonnes : Full, PCA 99%, PCA 95%
col_order = ['Full features'] + [c for c in pivot_acc.columns if 'PCA' in c]
col_order = [c for c in col_order if c in pivot_acc.columns]
pivot_acc = pivot_acc[col_order]
pivot_f1 = pivot_f1[col_order]

color_map = {'Full features': '#3498db'}
for c in col_order:
    if '99%' in c:
        color_map[c] = '#2ecc71'
    elif '95%' in c:
        color_map[c] = '#e74c3c'
bar_colors = [color_map.get(c, '#95a5a6') for c in col_order]

pivot_acc.plot(kind='barh', ax=axes[0], color=bar_colors, width=0.7)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('Accuracy')
axes[0].legend(loc='lower right', fontsize=9)

pivot_f1.plot(kind='barh', ax=axes[1], color=bar_colors, width=0.7)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('F1 Macro')
axes[1].legend(loc='lower right', fontsize=9)

plt.suptitle('Impact de la PCA sur les performances', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Tableau : temps d'entraînement ──
pivot_time = df_pca.pivot(index='Modèle', columns='Features', values='Temps (s)')[col_order]
print('\nTemps d\'entraînement (secondes) :')
pivot_time

## Entraînement des Modèles

In [ ]:
# ── Définition des modèles ──
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED,
        verbose=1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED,
        n_jobs=-1, verbose=1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', class_weight='balanced', probability=True,
        random_state=RANDOM_SEED, verbose=True
    ),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_SEED, verbose=1
    ),
}

# ── Entraînement et évaluation ──
results = []

for name, model in models.items():
    print(f'\n{"=" * 50}')
    print(f'Entraînement : {name}')
    print(f'{"=" * 50}')

    t0 = time.time()
    model.fit(X_train_s, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test_s)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append({
        'Modèle': name,
        'Accuracy': acc,
        'F1 Macro': f1,
        'Temps (s)': round(train_time, 1),
        'y_pred': y_pred,
    })

    print(f'Accuracy : {acc:.4f}  |  F1 Macro : {f1:.4f}  |  Temps : {train_time:.1f}s')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

## Comparaison des Modèles

In [ ]:
# ── Comparaison visuelle ──
df_results = pd.DataFrame(results).drop(columns='y_pred')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('viridis', len(df_results))

# Accuracy
bars1 = axes[0].barh(df_results['Modèle'], df_results['Accuracy'], color=colors)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('Accuracy')
axes[0].bar_label(bars1, fmt='%.4f', padding=3)

# F1 Macro
bars2 = axes[1].barh(df_results['Modèle'], df_results['F1 Macro'], color=colors)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('F1 Score (Macro)')
axes[1].bar_label(bars2, fmt='%.4f', padding=3)

plt.suptitle('Comparaison des Modèles ML (HOG + LBP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Modèles Supplémentaires

Ajout de classifieurs complémentaires pour élargir la comparaison :

| Modèle | Caractéristiques |
|--------|-----------------|
| **Extra Trees** | Variante de Random Forest avec splits aléatoires — souvent plus rapide et bon en généralisation |
| **AdaBoost** | Boosting séquentiel avec pondération des erreurs |
| **Bagging (SVM)** | Ensemble de SVM sur sous-échantillons — réduit la variance |
| **MLP** | Réseau de neurones classique (sklearn) — capture les non-linéarités complexes |
| **XGBoost** | Gradient boosting optimisé — régularisation L1/L2, très performant en ML tabulaire |
| **LightGBM** | Gradient boosting basé sur histogrammes — très rapide, gère bien les grands vecteurs |

In [ ]:
from sklearn.ensemble import (
    ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier
)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ── Modèles supplémentaires ──
extra_models = {
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED,
        n_jobs=-1, verbose=1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=200, random_state=RANDOM_SEED,
        algorithm='SAMME'
    ),
    'Bagging (SVM)': BaggingClassifier(
        estimator=SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_SEED),
        n_estimators=10, random_state=RANDOM_SEED, n_jobs=-1, verbose=1
    ),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(256, 128), max_iter=300, random_state=RANDOM_SEED,
        early_stopping=True, validation_fraction=0.1, verbose=True
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=1,
        eval_metric='mlogloss',
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, max_depth=-1, learning_rate=0.1,
        class_weight='balanced', random_state=RANDOM_SEED,
        n_jobs=-1, verbose=1,
    ),
}

for name, model in extra_models.items():
    print(f'\n{"=" * 50}')
    print(f'Entraînement : {name}')
    print(f'{"=" * 50}')

    t0 = time.time()
    model.fit(X_train_s, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test_s)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append({
        'Modèle': name,
        'Accuracy': acc,
        'F1 Macro': f1,
        'Temps (s)': round(train_time, 1),
        'y_pred': y_pred,
    })

    print(f'Accuracy : {acc:.4f}  |  F1 Macro : {f1:.4f}  |  Temps : {train_time:.1f}s')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# ── Mise à jour du classement ──
df_all = pd.DataFrame(results).drop(columns='y_pred').sort_values('F1 Macro', ascending=False)
df_all.index = range(1, len(df_all) + 1)
df_all.index.name = 'Rang'
df_all

## Optimisation des Hyperparamètres (RandomizedSearchCV)

On utilise `RandomizedSearchCV` plutôt que `GridSearchCV` exhaustif car le vecteur de features est large (~34k+). Cela permet d'explorer efficacement l'espace des hyperparamètres en un temps raisonnable.

**Stratégie** : on optimise les **3 meilleurs modèles** de la comparaison précédente.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform

# ── Grilles d'hyperparamètres par modèle ──
PARAM_GRIDS = {
    'Logistic Regression': {
        'model': LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED),
        'params': {
            'C': loguniform(1e-3, 1e3),
            'penalty': ['l1', 'l2'],
            'solver': ['saga'],
            'max_iter': [2000],
        },
        'n_iter': 20,
    },
    'Random Forest': {
        'model': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 400],
            'max_depth': [None, 20, 40, 60],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
        },
        'n_iter': 30,
    },
    'Extra Trees': {
        'model': ExtraTreesClassifier(class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 400],
            'max_depth': [None, 20, 40, 60],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
        },
        'n_iter': 30,
    },
    'SVM (RBF)': {
        'model': SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_SEED, probability=True),
        'params': {
            'C': loguniform(1e-1, 1e3),
            'gamma': ['scale', 'auto'] + list(loguniform(1e-5, 1e-1).rvs(5, random_state=RANDOM_SEED)),
        },
        'n_iter': 20,
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=RANDOM_SEED),
        'params': {
            'n_estimators': [100, 200, 400],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'min_samples_split': [2, 5, 10],
            'subsample': [0.8, 0.9, 1.0],
        },
        'n_iter': 30,
    },
    'KNN (k=5)': {
        'model': KNeighborsClassifier(n_jobs=-1),
        'params': {
            'n_neighbors': [3, 5, 7, 9, 11, 15],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan', 'cosine'],
        },
        'n_iter': 20,
    },
    'MLP': {
        'model': MLPClassifier(random_state=RANDOM_SEED, early_stopping=True, max_iter=300),
        'params': {
            'hidden_layer_sizes': [(128,), (256, 128), (512, 256), (256, 128, 64)],
            'alpha': loguniform(1e-5, 1e-1),
            'learning_rate_init': loguniform(1e-4, 1e-2),
            'activation': ['relu', 'tanh'],
        },
        'n_iter': 20,
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=RANDOM_SEED, n_jobs=-1, eval_metric='mlogloss'),
        'params': {
            'n_estimators': [100, 200, 400],
            'max_depth': [3, 5, 7, 9],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
        },
        'n_iter': 30,
    },
    'LightGBM': {
        'model': LGBMClassifier(class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1),
        'params': {
            'n_estimators': [100, 200, 400],
            'max_depth': [-1, 10, 20, 40],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'num_leaves': [31, 50, 80, 120],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
        },
        'n_iter': 30,
    },
}

# ── Sélectionner les 3 meilleurs modèles à optimiser ──
df_ranked = pd.DataFrame(results).drop(columns='y_pred').sort_values('F1 Macro', ascending=False)
top_models = df_ranked['Modèle'].head(3).tolist()
print(f'Top 3 modèles à optimiser : {top_models}\n')

# ── RandomizedSearchCV ──
optimized_results = []

for model_name in top_models:
    if model_name not in PARAM_GRIDS:
        print(f'⚠ Pas de grille définie pour {model_name}, skip.')
        continue

    cfg = PARAM_GRIDS[model_name]
    print(f'\n{"=" * 60}')
    print(f'Optimisation : {model_name} ({cfg["n_iter"]} itérations, CV=5)')
    print(f'{"=" * 60}')

    search = RandomizedSearchCV(
        cfg['model'],
        cfg['params'],
        n_iter=cfg['n_iter'],
        cv=5,
        scoring='f1_macro',
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
    )

    t0 = time.time()
    search.fit(X_train_s, y_train)
    search_time = time.time() - t0

    # Évaluer sur le test set
    y_pred_opt = search.best_estimator_.predict(X_test_s)
    acc_opt = accuracy_score(y_test, y_pred_opt)
    f1_opt = f1_score(y_test, y_pred_opt, average='macro')

    optimized_results.append({
        'Modèle': f'{model_name} (optimisé)',
        'Accuracy': acc_opt,
        'F1 Macro': f1_opt,
        'Temps (s)': round(search_time, 1),
        'y_pred': y_pred_opt,
        'best_params': search.best_params_,
        'cv_score': search.best_score_,
    })

    print(f'\nMeilleurs paramètres : {search.best_params_}')
    print(f'CV F1 Macro : {search.best_score_:.4f}')
    print(f'Test Accuracy : {acc_opt:.4f}  |  Test F1 Macro : {f1_opt:.4f}')
    print(f'Temps total : {search_time:.1f}s')
    print(classification_report(y_test, y_pred_opt, target_names=le.classes_))

In [ ]:
# ── Comparaison avant/après optimisation ──
# Combiner résultats de base + optimisés
all_results = results + optimized_results
df_compare = pd.DataFrame(all_results).drop(columns=['y_pred']).copy()
if 'best_params' in df_compare.columns:
    df_compare = df_compare.drop(columns=['best_params', 'cv_score'])
df_compare = df_compare.sort_values('F1 Macro', ascending=False).reset_index(drop=True)
df_compare.index += 1
df_compare.index.name = 'Rang'

# ── Visualisation ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2ecc71' if '(optimisé)' in m else '#3498db' for m in df_compare['Modèle']]

# Accuracy
bars1 = axes[0].barh(df_compare['Modèle'], df_compare['Accuracy'], color=colors)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('Accuracy')
axes[0].bar_label(bars1, fmt='%.4f', padding=3)

# F1 Macro
bars2 = axes[1].barh(df_compare['Modèle'], df_compare['F1 Macro'], color=colors)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('F1 Score (Macro)')
axes[1].bar_label(bars2, fmt='%.4f', padding=3)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', label='Baseline'),
                   Patch(facecolor='#2ecc71', label='Optimisé')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=12)

plt.suptitle('Comparaison Baseline vs Optimisé', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Tableau
print('\n' + '=' * 60)
print('CLASSEMENT FINAL — Tous modèles')
print('=' * 60)
df_compare

## Analyse du Meilleur Modèle

In [ ]:
# ── Sélection du meilleur modèle par F1 Macro (tous modèles inclus) ──
best_idx = np.argmax([r['F1 Macro'] for r in all_results])
best = all_results[best_idx]
print(f'Meilleur modèle : {best["Modèle"]} (F1 Macro = {best["F1 Macro"]:.4f})\n')

# Afficher les hyperparamètres si optimisé
if 'best_params' in best:
    print(f'Hyperparamètres : {best["best_params"]}\n')

# Classification report
print(classification_report(y_test, best['y_pred'], target_names=le.classes_))

# Matrice de confusion
cm = confusion_matrix(y_test, best['y_pred'])

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_
)
plt.title(f'Matrice de Confusion — {best["Modèle"]}')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.show()

## Analyse des Erreurs

In [ ]:
# ── Exemples mal classifiés ──
_, X_test_imgs, _, _ = train_test_split(
    images, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

misclassified = np.where(best['y_pred'] != y_test)[0]
n_show = min(16, len(misclassified))
indices = np.random.RandomState(RANDOM_SEED).choice(misclassified, n_show, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        idx = indices[i]
        ax.imshow(X_test_imgs[idx], cmap='gray')
        true_label = le.classes_[y_test[idx]]
        pred_label = le.classes_[best['y_pred'][idx]]
        ax.set_title(f'Vrai: {true_label}\nPrédit: {pred_label}', fontsize=9,
                     color='red' if true_label != pred_label else 'green')
    ax.axis('off')

plt.suptitle(f'Exemples mal classifiés — {best["Modèle"]}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTotal erreurs : {len(misclassified)} / {len(y_test)} '
      f'({len(misclassified)/len(y_test):.1%})')

## Tableau Récapitulatif

In [ ]:
# ── Résumé final ──
df_summary = pd.DataFrame(all_results).drop(columns=['y_pred'])
if 'best_params' in df_summary.columns:
    df_summary = df_summary.drop(columns=['best_params', 'cv_score'])
df_summary = df_summary.sort_values('F1 Macro', ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = 'Rang'

active = [k for k, v in FEATURES_CONFIG.items() if v]
print('\n' + '=' * 60)
print(f'RÉSUMÉ — Pipeline ML ({" + ".join(active)})')
print('=' * 60)
print(f'Features actives : {active}')
print(f'Vecteur total    : {X.shape[1]} features')
print(f'Train : {len(X_train)} | Test : {len(X_test)}')
print(f'Classes : {list(le.classes_)}')
print()
df_summary